# 02: Cointegration Research & Multiple-Testing FDR Control

**Objective**: Screen candidate pairs using return correlation, perform Engle-Granger two-step testing with MacKinnon asymptotic critical values, Johansen VECM rank estimation, and control for data snooping via Benjamini-Hochberg False Discovery Rate (FDR).

In [ ]:
from statarb.config import AppConfig
from statarb.research.candidate_generation import CandidatePairGenerator
from statarb.research.selection import PairSelector
from statarb.data.universe import UniverseFilter
import pandas as pd

cfg = AppConfig.from_yaml('../configs/default.yaml')
df = pd.read_parquet('../data/processed/prices.parquet')
pnl = df.pivot(index='date', columns='symbol', values='adj_close').sort_index()
returns = pnl.pct_change().dropna()

uf = UniverseFilter(cfg.universe)
cgen = CandidatePairGenerator(cfg.research.candidate_generation, uf.sector_map)
candidates = cgen.generate_candidates(returns)
print(f'Screened candidate pairs: {len(candidates)}')

In [ ]:
selector = PairSelector(cfg.research)
records = selector.evaluate_candidates(candidates, pnl.tail(252))
records_df = pd.DataFrame([r.to_dict() for r in records])
records_df[['pair_id', 'beta', 'raw_pvalue', 'fdr_qvalue', 'johansen_rank', 'is_selected', 'rejection_reason']]